# 필요 패키지 설치

In [ ]:
!pip install flask-ngrok flask pyngrok paho-mqtt opencv-python numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.9 MB/s eta 0:00:00


In [ ]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.3 MB/s eta 0:00:00


In [ ]:
!pip install pytz

# ngrok 주소, IP 확보

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("my_token")

In [ ]:
public_url = ngrok.connect(5000)
print("[ngrok 주소] ", public_url)

[ngrok 주소]  NgrokTunnel: "https://b656-34-16-170-228.ngrok-free.app" -> "http://localhost:5000"


In [ ]:
!nslookup b656-34-16-170-228.ngrok-free.app

Server:		127.0.0.11
Address:	127.0.0.11#53

Non-authoritative answer:
Name:	b656-34-16-170-228.ngrok-free.app
Address: 184.72.44.51
Name:	b656-34-16-170-228.ngrok-free.app
Address: 54.183.107.205
Name:	b656-34-16-170-228.ngrok-free.app
Address: 50.18.8.146
Name:	b656-34-16-170-228.ngrok-free.app
Address: 54.193.184.75
Name:	b656-34-16-170-228.ngrok-free.app
Address: 52.8.87.87
Name:	b656-34-16-170-228.ngrok-free.app
Address: 13.56.217.111
Name:	b656-34-16-170-228.ngrok-free.app
Address: 2600:1f1c:d8:5f01::6e:1
Name:	b656-34-16-170-228.ngrok-free.app
Address: 2600:1f1c:d8:5f00::6e:0
Name:	b656-34-16-170-228.ngrok-free.app
Address: 2600:1f1c:d8:5f01::6e:3
Name:	b656-34-16-170-228.ngrok-free.app
Address: 2600:1f1c:d8:5f00::6e:2
Name:	b656-34-16-170-228.ngrok-free.app
Address: 2600:1f1c:d8:5f01::6e:5
Name:	b656-34-16-170-228.ngrok-free.app
Address: 2600:1f1c:d8:5f00::6e:4



1. Jetson Nano에서 /etc/hosts 등록

    sudo nano /etc/hosts

    <IP 주소>    <ngrok 주소>

2. Fusion Node에 서버 주소 입력

    self.server_url = '<ngrok 주소>/upload'

In [ ]:
from flask import Flask, request, jsonify, send_from_directory, render_template_string
from flask_ngrok import run_with_ngrok
import os, base64, time
import numpy as np, cv2
from datetime import datetime
from pytz import timezone
from ultralytics import YOLO
import glob

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# === 초기 설정 ===
app = Flask(__name__)
os.makedirs("received_images", exist_ok=True)

# === YOLO 모델 로드 및 warm-up ===
model = YOLO('/content/drive/MyDrive/capstone/final_dataset_train/exp1/weights/best.pt')
_ = model.predict(source=np.zeros((224, 224, 3), dtype=np.uint8), verbose=False)

@app.route("/upload", methods=["POST"])
def upload():
    try:
        start_total = time.time()

        # === JSON 파싱 ===
        data = request.get_json(force=True, silent=True)
        if not data or 'image' not in data or 'timestamp' not in data:
            print("[ERROR] JSON 누락 필드 있음")
            return jsonify({"error": "Invalid data"}), 400

        image_b64 = data['image']
        timestamp_sent = float(data['timestamp'])

        # === 시간 측정 ===
        timestamp_recv = time.time()
        delay_transfer = timestamp_recv - timestamp_sent

        # === 이미지 디코딩 ===
        img_bytes = base64.b64decode(image_b64)
        img_np = np.frombuffer(img_bytes, np.uint8)
        image = cv2.imdecode(img_np, cv2.IMREAD_COLOR)

        # === YOLO 추론 ===
        start_infer = time.time()
        results = model.predict(source=image, verbose=False)[0]
        infer_time = time.time() - start_infer

        # === 시각화 및 저장 ===
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = f"Class {cls_id}: {conf:.2f}"
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        filename = datetime.fromtimestamp(timestamp_recv).isoformat().replace(":", "_") + ".jpg"
        cv2.imwrite(f"received_images/result_{filename}", image)

        # === 최종 시간 및 응답 ===
        total_time = time.time() - start_total
        print(f"[INFO] 전송 지연: {delay_transfer:.3f}초")
        print(f"[INFO] 추론 시간: {infer_time:.3f}초")
        print(f"[INFO] 전체 수신 - 응답 시간: {total_time:.3f}초")

        return jsonify({
            "status": "ok",
            "delay_transfer": delay_transfer,
            "inference_time": infer_time,
            "total_time": total_time
        }), 200

    except Exception as e:
        print("[ERROR] 처리 중 예외 발생:", str(e))
        return jsonify({"error": str(e)}), 500

# === Flask 서버 실행 ===
app.run(host='0.0.0.0', port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [26/May/2025 17:49:44] "POST /upload HTTP/1.1" 400 -


[ERROR] JSON 누락 필드 있음


INFO:werkzeug:127.0.0.1 - - [26/May/2025 17:52:03] "POST /upload HTTP/1.1" 200 -


[INFO] 전송 지연: 1.802초
[INFO] 추론 시간: 1.643초
[INFO] 전체 수신~응답 시간: 2.583초


...

[INFO] 전송 지연: 3.969초
[INFO] 추론 시간: 1.525초
[INFO] 전체 수신~응답 시간: 2.437초


INFO:werkzeug:127.0.0.1 - - [26/May/2025 17:52:20] "POST /upload HTTP/1.1" 200 -


[INFO] 전송 지연: 6.223초
[INFO] 추론 시간: 0.978초
[INFO] 전체 수신~응답 시간: 1.834초


INFO:werkzeug:127.0.0.1 - - [26/May/2025 17:52:22] "POST /upload HTTP/1.1" 200 -


[INFO] 전송 지연: 5.919초
[INFO] 추론 시간: 0.922초
[INFO] 전체 수신~응답 시간: 1.746초


# 전송 로직 비포함 Flask 서버

In [ ]:
# === 초기 설정 ===
app = Flask(__name__)
os.makedirs("received_images", exist_ok=True)

received_ids = []  # 전역으로 선언

MAX_IMAGE_FILES = 100  # 저장할 최대 이미지 수

def limit_image_files():
    image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")), key=os.path.getmtime)
    if len(image_files) > MAX_IMAGE_FILES:
        to_delete = image_files[:-MAX_IMAGE_FILES]
        for f in to_delete:
            try:
                os.remove(f)
            except Exception as e:
                print(f"[WARN] 이미지 삭제 실패: {f} - {e}")

# === YOLO 모델 로드 및 warm-up ===
model = YOLO('/content/best.pt')
_ = model.predict(source=np.zeros((224, 224, 3), dtype=np.uint8), verbose=False)

@app.route("/upload", methods=["POST"])
def upload():
    try:
        start_total = time.time()
        # === JSON 파싱 ===
        data = request.get_json(force=True, silent=True)
        if not data or 'image' not in data or 'timestamp' not in data:
            print("[ERROR] JSON 누락 필드 있음")
            return jsonify({"error": "Invalid data"}), 400

        image_b64 = data['image']
        timestamp_sent = float(data['timestamp'])

        frame_id = data.get("frame_id", -1)
        received_ids.append(frame_id)
        print(f"[RECV] 수신된 frame_id: {frame_id}")
        print(f"[RECV] 누적 수신 ID: {received_ids}")

        # 누락된 ID 탐지 (가장 최근 20개만 확인)
        if len(received_ids) >= 2:
            missing = sorted(set(range(min(received_ids), max(received_ids)+1)) - set(received_ids))
            if missing:
                print(f"[WARN] 누락된 frame_id 감지됨: {missing}")

        with open("frame_id_log.txt", "a") as f:
            f.write(f"{frame_id}\n")

        # === 시간 측정 ===
        timestamp_recv = time.time()
        delay_transfer = timestamp_recv - timestamp_sent

        # timestamp_str 없을 경우 → 한국 시간으로 포맷팅
    if timestamp_str is None:
        timestamp_str = datetime.fromtimestamp(timestamp_sent, KST).strftime('%Y-%m-%d %H:%M:%S')

        # === 이미지 디코딩 ===
        img_bytes = base64.b64decode(image_b64)
        img_np = np.frombuffer(img_bytes, np.uint8)
        image = cv2.imdecode(img_np, cv2.IMREAD_COLOR)

        # === YOLO 추론 ===
        start_infer = time.time()
        results = model.predict(source=image, verbose=False)[0]
        infer_time = time.time() - start_infer

        # === 포트홀이 탐지되지 않은 경우 처리 중단 ===
        if len(results.boxes) == 0:
            print("[INFO] 포트홀 미탐지 - 보고 생략")
            return jsonify({
                "status": "no_pothole",
                "message": "포트홀이 탐지되지 않아 보고하지 않음",
                "delay_transfer": delay_transfer,
                "inference_time": infer_time
            }), 200

        # === 시각화 및 저장 ===
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = f"Pothole: {conf:.2f}"
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        filename = datetime.fromtimestamp(timestamp_recv).isoformat().replace(":", "_") + ".jpg"
        cv2.imwrite(f"received_images/result_{filename}", image)
        limit_image_files()


        # === 최종 시간 및 응답 ===
        total_time = time.time() - start_total
        print(f"[INFO] 전송 지연: {delay_transfer:.3f}초")
        print(f"[INFO] 추론 시간: {infer_time:.3f}초")
        print(f"[INFO] 전체 수신 - 응답 시간: {total_time:.3f}초")

        # === 위치 정보 파싱 ===
        lidar_data = data.get("location", None)
        diameter = data.get("diameter", None)

        # === 위치 정보 출력 ===
        if lidar_data:
            print(f"[INFO] 위치 정보:")
            print(f"       - 상대 거리(relative_x): {lidar_data.get('relative_x')} m")
            print(f"       - 시작 거리(start_distance): {lidar_data.get('start_distance')} m")
            print(f"       - 종료 거리(end_distance): {lidar_data.get('end_distance')} m")
        else:
            print("[INFO] 위치 정보 없음")

        # === 전송 시각 (사람 읽기용) 출력 ===
        now_kst = datetime.now(KST).strftime('%Y-%m-%d %H:%M:%S')
        print(f"[INFO] 수신 시각: {data.get('now_kst')}")

        # === 시간 측정 이후에 추가 ===
        timestamp_str = data.get("timestamp_str", datetime.fromtimestamp(timestamp_sent).isoformat())


        if diameter is not None:
            severity = "위험"
        else:
            severity = "주의"
            diameter = None  # diameter는 위험 포트홀만 포함

        print(f"[INFO] 심각도 판단 결과: {severity}")

        return jsonify({
            "status": "ok",
            "severity": severity,
            "delay_transfer": delay_transfer,
            "inference_time": infer_time,
            "total_time": total_time,
            "diameter": diameter,
            "location": lidar_data
        }), 200

    except Exception as e:
        print("[ERROR] 처리 중 예외 발생:", str(e))
        return jsonify({"error": str(e)}), 500

# === Flask 서버 실행 ===
app.run(host='0.0.0.0', port=5000, threaded=True)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


[RECV] 수신된 frame_id: 0
[RECV] 누적 수신 ID: [0]


INFO:werkzeug:127.0.0.1 - - [29/May/2025 02:36:39] "POST /upload HTTP/1.1" 200 -


[INFO] 전송 지연: 2.006초
[INFO] 추론 시간: 0.930초
[INFO] 전체 수신 - 응답 시간: 1.520초
[INFO] 위치 정보:
       - 상대 거리(relative_x): 115.13875579833984 m
       - 시작 거리(start_distance): -1.0 m
       - 종료 거리(end_distance): -1.0 m
[INFO] 수신 시각 (ISO 포맷): 2025-05-29T04:36:36.534031
[INFO] 심각도 판단 결과: 주의
[RECV] 수신된 frame_id: 1
[RECV] 누적 수신 ID: [0, 1]


...

[INFO] 전송 지연: 2.910초
[INFO] 추론 시간: 0.844초
[INFO] 전체 수신 - 응답 시간: 1.576초
[INFO] 위치 정보:
       - 상대 거리(relative_x): 219.51026916503906 m
       - 시작 거리(start_distance): -1.0 m
       - 종료 거리(end_distance): -1.0 m
[INFO] 수신 시각 (ISO 포맷): 2025-05-29T04:45:07.923762
[INFO] 심각도 판단 결과: 주의
[RECV] 수신된 frame_id: 187
[RECV] 누적 수신 ID: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 1

INFO:werkzeug:127.0.0.1 - - [29/May/2025 02:45:14] "POST /upload HTTP/1.1" 200 -


[INFO] 전송 지연: 3.384초
[INFO] 추론 시간: 0.791초
[INFO] 전체 수신 - 응답 시간: 1.646초
[INFO] 위치 정보:
       - 상대 거리(relative_x): 219.9182891845703 m
       - 시작 거리(start_distance): -1.0 m
       - 종료 거리(end_distance): -1.0 m
[INFO] 수신 시각 (ISO 포맷): 2025-05-29T04:45:09.924345
[INFO] 심각도 판단 결과: 주의
[RECV] 수신된 frame_id: 188
[RECV] 누적 수신 ID: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 15

INFO:werkzeug:127.0.0.1 - - [29/May/2025 02:45:17] "POST /upload HTTP/1.1" 200 -


[INFO] 전송 지연: 3.973초
[INFO] 추론 시간: 0.815초
[INFO] 전체 수신 - 응답 시간: 2.293초
[INFO] 위치 정보:
       - 상대 거리(relative_x): 220.44871520996094 m
       - 시작 거리(start_distance): -1.0 m
       - 종료 거리(end_distance): -1.0 m
[INFO] 수신 시각 (ISO 포맷): 2025-05-29T04:45:12.525280
[INFO] 심각도 판단 결과: 주의


# 전송 로직 포함 Flask 서버

In [ ]:
!pkill -f ngrok

In [ ]:
# === 초기 설정 ===
app = Flask(__name__)
run_with_ngrok(app)  # ngrok과 Flask 연결
IMAGE_DIR = "received_images"
os.makedirs(IMAGE_DIR, exist_ok=True)
KST = timezone('Asia/Seoul')
received_logs = []

def limit_image_files():
    image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")), key=os.path.getmtime)
    if len(image_files) > 300:
        to_delete = image_files[:-300]
        for f in to_delete:
            os.remove(f)

# === YOLO 로딩 ===
model = YOLO('/content/best.pt')
_ = model.predict(source=np.zeros((224, 224, 3), dtype=np.uint8), verbose=False)

# === 업로드 엔드포인트 ===
@app.route("/upload", methods=["POST"])
def upload():
    try:
        start_total = time.time()
        data = request.get_json(force=True, silent=True)
        if not data or 'image' not in data or 'timestamp' not in data:
            return jsonify({"error": "Invalid data"}), 400

        image_b64 = data['image']
        timestamp_sent = float(data['timestamp'])
        lidar_data = data.get("location", None)
        diameter = data.get("diameter", None)
        timestamp_str = datetime.fromtimestamp(timestamp_sent, KST).strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

        # 디코딩
        img_bytes = base64.b64decode(image_b64)
        img_np = np.frombuffer(img_bytes, np.uint8)
        image = cv2.imdecode(img_np, cv2.IMREAD_COLOR)

        # === YOLO 추론 ===
        start_infer = time.time()
        results = model.predict(source=image, verbose=False)[0]
        infer_time = time.time() - start_infer

        # === YOLO 탐지 여부 확인
        yolo_detected = len(results.boxes) > 0

        # === 시각화 및 저장 ===
        filename = None
        if yolo_detected:
            # 시각화: 박스 그리기
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                conf = float(box.conf[0])
                label = f"Pothole: {conf:.2f}"
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # 이미지 저장 조건을 확장함
        if yolo_detected or (lidar_data and lidar_data.get("start_distance", -1) > 0):
            filename = datetime.now(KST).strftime("%Y%m%d_%H%M%S") + ".jpg"
            save_path = os.path.join(IMAGE_DIR, filename)
            cv2.imwrite(save_path, image)
            limit_image_files()

        # === 심각도 판단 로직 ===
        lidar_detected = lidar_data and lidar_data.get("start_distance", -1) > 0

        if yolo_detected and lidar_detected:
            severity = "위험"
        elif yolo_detected:
            severity = "주의"
        else:
            return jsonify({"status": "no_detection"}), 200

        now_kst = datetime.now(KST)

        # 로그 저장 (최근 100개만 유지)
        received_logs.append({
            "timestamp": timestamp_str,
            "severity": severity,
            "location": lidar_data,
            "filename": filename,
            "received_time": now_kst,
            "diameter": diameter
        })
        if len(received_logs) > 100:
            received_logs.pop(0)

        return jsonify({
            "status": "ok",
            "severity": severity,
            "delay_transfer": time.time() - timestamp_sent,
            "inference_time": infer_time,
            "total_time": time.time() - start_total,
            "diameter": diameter,
            "location": lidar_data
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# === 대시보드 페이지 ===
@app.route("/dashboard")
def dashboard():
    now = datetime.now(KST)  # 대시보드 렌더링 시간
    html = """
    <html>
    <head>
        <meta http-equiv="refresh" content="3">
        <title>포트홀 보고 로그</title>
    </head>
    <body>
        <h2>[실시간 포트홀 보고 로그]</h2>
        {% for log in logs[::-1] %}
            <div style="margin-bottom:20px; border-bottom:1px solid #ccc;">
                <p><b>시각:</b> {{ log.timestamp | safe }}</p>
                <p><b>심각도:</b> {{ log.severity }}</p>
                {% if log.location %}
                    {% if log.severity == "위험" and log.diameter %}
                        <p><b>지름:</b> {{ "%.2f"|format(log.diameter) }} m</p>
                    {% endif %}
                    {% set rel = log.location.get('relative_x', -1) %}
                    {% set start = log.location.get('start_distance', -1) %}
                    {% set end = log.location.get('end_distance', -1) %}
                    {% if rel > -0.1 %}
                        <p><b>위치:</b><br>
                        &nbsp;&nbsp;- 상대 거리: {{ "%.2f"|format(rel) }} m<br>
                        &nbsp;&nbsp;- 시작: {{ "%.2f"|format(start) if start > -0.1 else "-" }}<br>
                        &nbsp;&nbsp;- 종료: {{ "%.2f"|format(end) if end > -0.1 else "-" }}
                        </p>
                    {% endif %}
                {% endif %}
                {# {% if log.received_time %}
                    {% set elapsed = (now - log.received_time).total_seconds() %}
                    <p><b>출력까지 경과 시간:</b> {{ "%.2f"|format(elapsed) }}초</p>
                {% endif %} #}
                {% if log.filename %}
                    <img src="/images/{{ log.filename }}" width="400"><br>
                {% endif %}
            </div>
        {% endfor %}
    </body></html>
    """
    return render_template_string(html, logs=received_logs, now=now)

# === 이미지 서빙 ===
@app.route("/images/<filename>")
def images(filename):
    return send_from_directory(IMAGE_DIR, filename)

# === 서버 실행 ===
if __name__ == "__main__":
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


 * Running on http://83bf-34-45-238-218.ngrok-free.app
 * Traffic stats available on http://127.0.0.1:4040


INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:00] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:00] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:03] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:08] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:12] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:15] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:19] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:23] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:27] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:29] "POST /upload HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:30] "GET /dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/May/2025 15:05:33] "POST /upload HTTP/1.1" 200 -
INFO:w

In [ ]:
import os
import glob

IMAGE_DIR = "received_images"

def delete_all_images():
    files = glob.glob(os.path.join(IMAGE_DIR, "*"))
    for f in files:
        try:
            os.remove(f)
            print(f"[삭제됨] {f}")
        except Exception as e:
            print(f"[오류] {f} 삭제 실패: {e}")

delete_all_images()


[삭제됨] received_images/2025-06-12T21:07:53.959309.jpg
[삭제됨] received_images/2025-06-12T21:28:37.893441.jpg
[삭제됨] received_images/2025-06-12T21:01:00.163143.jpg
[삭제됨] received_images/2025-06-12T21:18:25.895231.jpg
[삭제됨] received_images/2025-06-12T21:17:45.897789.jpg
[삭제됨] received_images/2025-06-12T21:20:56.695459.jpg
[삭제됨] received_images/2025-06-12T21:06:08.362880.jpg
[삭제됨] received_images/2025-06-12T20:56:27.658388.jpg
[삭제됨] received_images/2025-06-12T21:26:53.093130.jpg
[삭제됨] received_images/2025-06-12T21:07:17.964164.jpg
[삭제됨] received_images/2025-06-12T21:19:49.902677.jpg
[삭제됨] received_images/2025-06-12T21:20:44.291773.jpg
[삭제됨] received_images/2025-06-12T21:00:56.063240.jpg
[삭제됨] received_images/2025-06-12T21:18:09.895676.jpg
[삭제됨] received_images/2025-06-12T20:53:51.858845.jpg
[삭제됨] received_images/2025-06-12T21:03:56.464697.jpg
[삭제됨] received_images/2025-06-12T20:39:31.597830.jpg
[삭제됨] received_images/2025-06-12T21:24:17.696317.jpg
[삭제됨] received_images/2025-06-12T21:27:22.7026

# Jetson Nano에서 추론하는 Flask 서버

In [ ]:
app = Flask(__name__)
SAVE_DIR = "received_images"
os.makedirs(SAVE_DIR, exist_ok=True)

@app.route("/upload", methods=["POST"])
def upload():
    try:
        start_time = time.time()

        data = request.get_json(force=True)
        timestamp_sent = float(data["timestamp"])
        image_id = data["image_id"]
        img_b64 = data["image"]
        yolo_start = float(data["yolo_start_time"])
        yolo_end = float(data["yolo_end_time"])
        sync_diff = float(data["sync_time_diff"])
        detections = data.get("detections", [])

        transmission_delay = start_time - timestamp_sent
        total_elapsed = time.time() - yolo_start
        yolo_infer_time = yolo_end - yolo_start

        # 저장
        img_bytes = base64.b64decode(img_b64)
        img_path = os.path.join(SAVE_DIR, f"{image_id}.jpg")
        with open(img_path, "wb") as f:
            f.write(img_bytes)

        # 로그 출력
        print(f"[RECV] 이미지 ID: {image_id}")
        print(f"       YOLO 추론 시간: {yolo_infer_time:.3f}초")
        print(f"       동기화 시간차: {sync_diff:.3f}초")
        print(f"       탐지 수: {len(detections)}")
        print(f"       전송 지연: {transmission_delay:.3f}초")
        print(f"       전체 파이프라인 처리 시간: {total_elapsed:.3f}초")

        return jsonify({
            "status": "ok",
            "transmission_delay": transmission_delay,
            "total_elapsed": total_elapsed
        }), 200

    except Exception as e:
        print(f"[ERROR] 처리 중 오류 발생: {e}")
        return jsonify({"status": "error"}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [12/Jun/2025 22:40:16] "POST /upload HTTP/1.1" 200 -


[RECV] 이미지 ID: 2025-06-12T22:40:12.472640
       YOLO 추론 시간: 0.260초
       동기화 시간차: 0.071초
       탐지 수: 1
       전송 지연: 0.519초
       전체 파이프라인 처리 시간: 2.234초


INFO:werkzeug:127.0.0.1 - - [12/Jun/2025 22:40:19] "POST /upload HTTP/1.1" 200 -


[RECV] 이미지 ID: 2025-06-12T22:40:14.874835
       YOLO 추론 시간: 0.266초
       동기화 시간차: 0.068초
       탐지 수: 1
       전송 지연: 0.507초
       전체 파이프라인 처리 시간: 2.553초


...

[RECV] 이미지 ID: 2025-06-12T22:43:15.471450
       YOLO 추론 시간: 0.256초
       동기화 시간차: 0.091초
       탐지 수: 1
       전송 지연: 0.560초
       전체 파이프라인 처리 시간: 2.053초


INFO:werkzeug:127.0.0.1 - - [12/Jun/2025 22:43:21] "POST /upload HTTP/1.1" 200 -


[RECV] 이미지 ID: 2025-06-12T22:43:17.871219
       YOLO 추론 시간: 0.260초
       동기화 시간차: 0.120초
       탐지 수: 1
       전송 지연: 0.494초
       전체 파이프라인 처리 시간: 1.899초


INFO:werkzeug:127.0.0.1 - - [12/Jun/2025 22:43:24] "POST /upload HTTP/1.1" 200 -


[RECV] 이미지 ID: 2025-06-12T22:43:20.072595
       YOLO 추론 시간: 0.256초
       동기화 시간차: 0.121초
       탐지 수: 1
       전송 지연: 0.640초
       전체 파이프라인 처리 시간: 2.169초
